# 6. Solenoid ON and OFF

Perform one deliberate ON/OFF pair through the qualified owner, with explicit cleanup.


## Before you start

Follow [the notebook setup and safety guide](README.md), then select the
project's Python 3.12 kernel. Run cells from top to bottom. Each notebook
is independent. Default execution uses a fictional config and makes no
hardware calls; a skipped operation is **not a successful hardware test**.

For an authorized live session, set `CONFIG` to `ROOT / 'shutters.local.toml'`,
set `TARGET` to the reviewed exact device name, and only then enable the
relevant gate. Keep the source independently protected. Release any GUI,
CLI or other notebook owner first. Restart the kernel after changing setup.


In [ ]:
from pathlib import Path
from dataclasses import asdict
from pprint import pprint
from thorlabs_shutter_control import ShutterHub, load_config

# Works from the repository root or docs/examples. No device access.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'config/shutters.example.toml').is_file()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from inside the project checkout.')
CONFIG = ROOT / 'config/shutters.example.toml'  # Fictional; offline only.
TARGET = 'example'  # Exact configured name; never select the first device.
RUN_HARDWARE = False

configs = load_config(CONFIG)
selected = next((c for c in configs if c.name == TARGET), None)
if selected is None:
    raise ValueError('TARGET must match an exact configured name.')
print('Selected:', selected.name, selected.serial_number, selected.route.kind)


## 1. Required physical prerequisites

This is an actuation example. Before enabling it, establish an approved
bounded window for this exact serial/route, actual shutter and wiring,
independently protected beam/source, interlocks, permitted initial state,
load ratings and cleanup. No current project deployment is physically
qualified. A gate value does not grant authorization.

Use an approved continuous-duty manual load with an independently
demonstrated OFF → optical closed mapping and reviewed minimum OFF time.
Begin in Manual/OFF. This notebook will **not** change mode. VCP is read-only;
KEH3 is unavailable. Do not use `enable_device()` as initialization: it
aliases output and is deliberately rejected.


In [ ]:
RUN_OUTPUT = False
print('Actuation gate:', RUN_HARDWARE and RUN_OUTPUT)
print('Qualification record:', selected.policy.evidence if selected.policy else 'ABSENT')


## 2. One ON request, then OFF in the same cell

`set_solenoid(TARGET, True)` requests electrical ON; `False` requests
qualified priority OFF. Never leave ON and OFF in separate notebook cells.
This pair has no calibrated exposure time and no repeated cycles. For a
host-timed hold the API provides `pulse(target, seconds)`, but it does not
measure optical exposure or provide a hard safety cutoff.

The `finally` block attempts OFF even after an error or normal keyboard
interrupt. If both operation and cleanup fail, the original exception and
cleanup detail remain visible. Kernel termination, process failure or a
broken link can defeat cleanup; independent source protection is essential.


In [ ]:
if RUN_HARDWARE and RUN_OUTPUT:
    if selected.policy is None:
        raise RuntimeError('No qualified policy. Do not invent one to enable this cell.')
    hub = ShutterHub(configs)
    connected = False
    original_error = None
    try:
        hub.connect(TARGET)
        connected = True
        before = hub.get_status(TARGET)
        if (before.feedback.operating_mode != 'manual'
                or before.feedback.electrical_state != 'off'):
            raise RuntimeError('Expected Manual/OFF. Stop and review the physical setup.')
        on_result = hub.set_solenoid(TARGET, True)
        print('ON electrical readback:', on_result.feedback.electrical_state)
    except BaseException as exc:
        original_error = exc
        raise
    finally:
        try:
            if connected:
                try:
                    off_result = hub.set_solenoid(TARGET, False)
                    print('OFF electrical readback:', off_result.feedback.electrical_state)
                    print('Optical state:', off_result.feedback.shutter_state)
                except BaseException as cleanup_error:
                    print('OFF UNCONFIRMED: independently inhibit source; optical state unknown.')
                    if original_error is None:
                        raise
                    original_error.add_note(f'Notebook OFF cleanup failed: {cleanup_error!r}')
        finally:
            try:
                hub.close()  # Communication release only, never proof of closure.
            except BaseException as release_error:
                print('Communication release failed:', repr(release_error))
                raise
else:
    print('ON/OFF skipped; no output command sent.')


## 3. Evaluate and stop

Successful electrical readback is `on`, then `off`. Independently observe
the blade/beam with the approved method and record that result separately.
ON/OFF alone does not establish switching time, duty rating or interlocks.
`open_shutter()` / `close_shutter()` use the same qualification boundary;
their names do not add an optical sensor.

After any ambiguous output, preserve the error and cleanup notes; do not
rerun ON. Persistent lockout survives reconnect/process restart. With the
source independently inhibited, restore the exact controller under the
[protected recovery procedure](../python-api.md). Do not delete owner or
uncertainty records. On partial-write framing loss, even OFF cannot safely
be sent on that connection. A successful OFF does not clear an old fault.
